# Konverzija težina u Keras modele
Pokreni ćelije redom. Ne trenira modele; izvorne težine ostaju netaknute.

In [1]:
# Konverzija best_model.weights.h5 u best_model.keras za evaluate.ipynb.
# Python 3.8 kompatibilno; rekurzivno mapiranje težina po putanji sloja.

import os
import sys
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
sys.modules["jax"] = None
import gc
import json
import math
import warnings
import h5py
from pathlib import Path

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
except ImportError as exc:
    raise SystemExit(f"GREŠKA: TensorFlow nije dostupan: {exc}") from exc

BASE_PATH = Path("/home/fipulab/anastazija/TEyeD_preprocessed")
ROOT = BASE_PATH / "128x128_stride10_q4_seg_pupil-iris-lid_2D"
RUNS = Path("/home/fipulab/anastazija/runs")
MANDATORY_CONFIGS = ["mtl_full", "mtl_wseg0", "mtl_pupil"]
MANDATORY_SEEDS = [42, 123, 2024]
GAZE_SEEDS = [42, 123, 2024, 7, 13]
PARTS = ["pupil", "iris", "lid"]
DENSE_UNITS = 256
DROPOUT = 0.3

GPUS = tf.config.list_physical_devices("GPU")
for gpu in GPUS:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as exc:
        warnings.warn(f"GPU memory growth nije postavljen: {exc}")
keras.mixed_precision.set_global_policy("mixed_float16" if GPUS else "float32")
print(f"TensorFlow {tf.__version__}; GPU: {len(GPUS)}; policy: {keras.mixed_precision.global_policy()}")


@keras.utils.register_keras_serializable(package='MTL')
def unit_gaze(x): return tf.math.l2_normalize(tf.cast(x, tf.float32), axis=-1)

@keras.utils.register_keras_serializable(package='MTL')
def gaze_cosine(y_true, y_pred):
    y_true = unit_gaze(y_true); y_pred = unit_gaze(y_pred)
    return 1. - tf.reduce_sum(y_true * y_pred, axis=-1)

@keras.utils.register_keras_serializable(package='MTL')
def mean_angle_error(y_true, y_pred):
    dot = tf.reduce_sum(unit_gaze(y_true) * unit_gaze(y_pred), axis=-1)
    return tf.acos(tf.clip_by_value(dot, -1., 1.)) * 180. / math.pi

@keras.utils.register_keras_serializable(package='MTL')
class MaskedBCE(keras.losses.Loss):
    def __init__(self, parts, name='masked_bce', **kwargs):
        super().__init__(name=name, **kwargs); self.parts = list(parts)
    def call(self, y_true, y_pred):
        ids = [PARTS.index(p) for p in self.parts]
        truth = tf.gather(y_true[..., :3], ids, axis=-1)
        valid = tf.gather(y_true[..., 3:], ids, axis=-1)
        pred = tf.gather(tf.cast(y_pred, tf.float32), ids, axis=-1)
        per = keras.backend.binary_crossentropy(truth, pred) * valid
        return tf.reduce_sum(per, axis=[1, 2, 3]) / tf.maximum(tf.reduce_sum(valid, axis=[1, 2, 3]), 1.)
    def get_config(self): return {**super().get_config(), 'parts': self.parts}

@keras.utils.register_keras_serializable(package='MTL')
class PartIoU(keras.metrics.Metric):
    def __init__(self, part, **kwargs):
        super().__init__(name='iou_' + part, **kwargs); self.part = part; self.idx = PARTS.index(part)
        self.inter = self.add_weight(name='intersection', initializer='zeros')
        self.union = self.add_weight(name='union', initializer='zeros')
    def update_state(self, y_true, y_pred, sample_weight=None):
        i = self.idx; valid = y_true[..., i + 3] > .5
        truth = tf.logical_and(y_true[..., i] > .5, valid)
        pred = tf.logical_and(tf.cast(y_pred[..., i], tf.float32) > .5, valid)
        self.inter.assign_add(tf.reduce_sum(tf.cast(truth & pred, tf.float32)))
        self.union.assign_add(tf.reduce_sum(tf.cast(truth | pred, tf.float32)))
    def result(self): return tf.math.divide_no_nan(self.inter, self.union)
    def reset_state(self): self.inter.assign(0.); self.union.assign(0.)
    def get_config(self): return {**super().get_config(), 'part': self.part}


def build_mtl(seed):
    path = RUNS / 'seg_only' / f'seed_{seed}' / 'best_parts.keras'
    if not path.exists(): raise FileNotFoundError(f'Nedostaje predtrenirani seg model: {path}')
    original_seg = keras.models.load_model(path, compile=False)
    original_encoder = original_seg.get_layer('feature_extractor')

    def clone_encoder_layer(layer):
        cfg = layer.get_config()
        if isinstance(layer, layers.BatchNormalization):
            if 'fused' in cfg: cfg['fused'] = False
        return layer.__class__.from_config(cfg)
    nonfused_encoder = keras.models.clone_model(original_encoder, clone_function=clone_encoder_layer)

    def clone_seg_layer(layer):
        if layer.name == 'feature_extractor': return nonfused_encoder
        return layer.__class__.from_config(layer.get_config())
    seg = keras.models.clone_model(original_seg, clone_function=clone_seg_layer)
    seg.set_weights(original_seg.get_weights())

    for bn in nonfused_encoder.submodules:
        if isinstance(bn, layers.BatchNormalization):
            bn.fused = False
            if hasattr(bn, '_fused'): bn._fused = False
            if getattr(bn, 'fused', False) or getattr(bn, '_fused', False):
                raise RuntimeError(f'Ne mogu isključiti fused BatchNormalization: {bn.name}')

    if seg.output_shape[-1] != 3: raise ValueError(f'{path}: očekivana 3 segmentacijska kanala')
    encoder = seg.get_layer('feature_extractor')
    del original_seg, original_encoder
    gc.collect()

    seg_convs = [(i, l) for i, l in enumerate(seg.layers) if isinstance(l, layers.Conv2D) and l.filters == 3]
    if not seg_convs: raise ValueError('Nema završnog Conv2D(filters=3).')
    seg_conv_idx, seg_conv = seg_convs[-1]
    candidates = [l for l in seg.layers[:seg_conv_idx]
                  if isinstance(l, (layers.Conv2DTranspose, layers.UpSampling2D, layers.Resizing))]
    if not candidates: raise ValueError(f'Nema dekoderskih značajki prije {seg_conv.name}.')
    decoder_layer = candidates[-1]; decoder_features = decoder_layer.output
    print('decoder_features:', decoder_layer.name, decoder_features.shape)

    try:
        enc_outputs = encoder.get_output_at(-1)
    except (AttributeError, TypeError):
        enc_outputs = encoder.output
    outputs = list(enc_outputs) if isinstance(enc_outputs, (list, tuple)) else [enc_outputs]
    spatial = []
    for tensor in outputs:
        shape = tensor.shape
        if len(shape) != 4: continue
        h, w = shape[1], shape[2]
        if h is not None and w is not None: spatial.append((int(h) * int(w), tensor))
    if not spatial: raise ValueError('Enkoder nema 4D izlaz.')
    bottleneck = min(spatial, key=lambda item: item[0])[1]
    h, w = bottleneck.shape[1:3]
    if int(h) >= 128 or int(w) >= 128:
        raise ValueError(f'Bottleneck nije smanjen ispod 128.')
    print('encoder_features:', encoder.name, bottleneck.shape)

    seg_gap = layers.GlobalAveragePooling2D(name='fusion_seg_gap')(decoder_features)
    enc_gap = layers.GlobalAveragePooling2D(name='fusion_encoder_gap')(bottleneck)
    fused = layers.Concatenate(name='fusion_concat')([enc_gap, seg_gap])
    fused = layers.Dense(128, activation='relu', name='fusion_dense1')(fused)
    fused = layers.Dropout(.3, name='fusion_dropout')(fused)
    fused = layers.Dense(64, activation='relu', name='fusion_dense2')(fused)
    raw = layers.Dense(3, name='gaze_raw', dtype='float32')(fused)
    gaze = layers.Lambda(unit_gaze, name='gaze_output', dtype='float32')(raw)
    model = keras.Model(seg.input, {'seg': seg.output, 'gaze_output': gaze},
                        name='MTL_CrossFeatureFusion')
    return model, encoder


@keras.utils.register_keras_serializable(package="GazeOnly")
def gaze_unit_normalize(t):
    return tf.math.l2_normalize(t, axis=-1)


def build_gaze_model():
    inputs = keras.Input(shape=(128, 128, 3), name="image")
    encoder = keras.applications.MobileNetV3Small(
        input_shape=(128, 128, 3), include_top=False, weights="imagenet",
        include_preprocessing=False
    )
    x = encoder(inputs)
    x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = layers.Dense(DENSE_UNITS, activation="relu", name="gaze_dense")(x)
    x = layers.Dropout(DROPOUT, name="gaze_dropout")(x)
    raw = layers.Dense(3, dtype="float32", name="gaze_raw")(x)
    output = layers.Lambda(gaze_unit_normalize,
                           name="gaze_unit_vector", dtype="float32")(raw)
    model = keras.Model(inputs, output, name="MobileNetV3Small_GazeOnly")
    return model, encoder


def custom_mtl_objects():
    return {'unit_gaze': unit_gaze, 'gaze_cosine': gaze_cosine, 'mean_angle_error': mean_angle_error,
            'MaskedBCE': MaskedBCE, 'PartIoU': PartIoU,
            'MTL>unit_gaze': unit_gaze, 'MTL>gaze_cosine': gaze_cosine,
            'MTL>mean_angle_error': mean_angle_error, 'MTL>MaskedBCE': MaskedBCE, 'MTL>PartIoU': PartIoU,
            'gaze_unit_normalize': gaze_unit_normalize,
            'GazeOnly>gaze_unit_normalize': gaze_unit_normalize}


def _strip_colon0(name):
    """Python 3.8 kompatibilna zamjena za str.removesuffix(':0')."""
    return name.rsplit(":", 1)[0] if name.endswith(":0") else name


def _iter_leaf_layers_with_path(model, prefix=''):
    """Rekurzivno vraća (leaf_layer, full_path) za sve listove — preskače nested keras.Model."""
    for layer in model.layers:
        path = f'{prefix}/{layer.name}' if prefix else layer.name
        if isinstance(layer, keras.Model):
            # Nested model: ne obrađuj .weights direktno, uđi rekurzivno.
            yield from _iter_leaf_layers_with_path(layer, path)
        else:
            yield layer, path


def load_weights_by_layer_paths(model, weights_path, verbose=False):
    """Rekurzivno mapira H5 datasete na modelove leaf težine po putanji sloja."""
    h5_flat = {}

    def walk(group, prefix=""):
        for key, item in group.items():
            path = f"{prefix}/{key}" if prefix else key
            if isinstance(item, h5py.Dataset):
                clean = _strip_colon0(path)
                if clean in h5_flat:
                    raise ValueError(f"Duplicirana H5 putanja: {clean}")
                h5_flat[clean] = item[()]
            elif isinstance(item, h5py.Group):
                walk(item, path)

    with h5py.File(str(weights_path), "r") as f:
        walk(f)

    if verbose:
        print(f"  H5 dataseta: {len(h5_flat)}")

    assignments = []
    used_h5 = set()
    missing = []

    for layer, layer_path in _iter_leaf_layers_with_path(model):
        if not layer.weights:
            continue

        for w in layer.weights:
            w_short = _strip_colon0(w.name).split("/")[-1]
            candidates = [
                f'{layer_path}/{w_short}',
                f'{layer_path}/{layer.name}/{w_short}',
                f'{layer.name}/{w_short}',
                f'{layer.name}/{layer.name}/{w_short}',
            ]
            found = None
            for c in candidates:
                if c in h5_flat:
                    found = c
                    break

            if found is None:
                want_shape = tuple(int(d) for d in w.shape)
                shape_matches = [p for p, v in h5_flat.items()
                                 if v.shape == want_shape
                                 and p.rsplit('/', 1)[-1] == w_short]
                layer_token_matches = [p for p in shape_matches if layer.name in p.split('/')]
                if len(layer_token_matches) == 1:
                    found = layer_token_matches[0]
                elif len(shape_matches) == 1:
                    found = shape_matches[0]

            if found is None:
                missing.append((layer_path, layer.name, w_short, tuple(int(d) for d in w.shape)))
                continue

            value = h5_flat[found]
            expected = tuple(int(d) for d in w.shape)
            if value.shape != expected:
                raise ValueError(
                    f"{layer_path}/{w_short}: oblik se ne poklapa "
                    f"(H5: {found} {value.shape}, model: {expected})")
            if found in used_h5:
                raise ValueError(f"H5 dataset {found} već iskorišten za drugu težinu.")
            assignments.append((w, value, layer_path, w_short, found))
            used_h5.add(found)

    if missing:
        lines = []
        for layer_path, layer_name, w_short, shape in missing[:15]:
            lines.append(f"  - {layer_path} :: {w_short} {shape}")
        raise ValueError(
            f"Ne mogu mapirati {len(missing)} težina. Primjeri:\n" + "\n".join(lines) +
            f"\nDostupno H5 (prvih 15): {list(h5_flat)[:15]}")

    for w, value, layer_path, w_short, found in assignments:
        w.assign(value)
        if verbose:
            print(f"  {layer_path}/{w_short} <- {found} {value.shape}")

    print(f"  Postavljeno težina: {len(assignments)}/{len(model.weights)}")
    unused = sorted(set(h5_flat) - used_h5)
    if unused:
        print(f"  Napomena: {len(unused)} dodatnih H5 dataseta nije korišteno: {unused[:10]}")
    return len(assignments)

def convert_one(run_dir, arch_builder, custom_objects):
    """Vraća status 'konvertirano', 'preskočeno' ili 'neuspješno'."""
    weights_path = run_dir / "best_model.weights.h5"
    out_path = run_dir / "best_model.keras"
    if out_path.is_file():
        print(f"Već postoji, preskačem: {out_path}")
        return "preskočeno"
    if not weights_path.is_file():
        warnings.warn(f"Nedostaju težine, preskačem: {weights_path}")
        return "preskočeno"

    model = None
    try:
        with keras.utils.custom_object_scope(custom_objects):
            built = arch_builder()
            model = built[0] if isinstance(built, tuple) else built
            load_weights_by_layer_paths(model, weights_path, verbose=True)
            out_test = model(tf.zeros([1, 128, 128, 3]), training=False)
            if isinstance(out_test, dict):
                shapes = {k: tuple(v.shape) for k, v in out_test.items()}
            elif isinstance(out_test, (list, tuple)):
                shapes = [tuple(v.shape) for v in out_test]
            else:
                shapes = tuple(out_test.shape)
            print(f"  Test forward pass OK, izlazi: {shapes}")
            temp_path = run_dir / "best_model.tmp.keras"
            try:
                model.save(str(temp_path))
            except (AttributeError, TypeError) as exc:
                warnings.warn(f"Spremanje modela nije uspjelo ({exc}); pokušavam bez optimizatora.")
                try:
                    model.save(str(temp_path), include_optimizer=False)
                except (AttributeError, TypeError, ValueError) as second:
                    warnings.warn(f"Rezervno spremanje nije uspjelo: {second}")
                    (run_dir / "best_model.architecture.json").write_text(
                        model.to_json(), encoding="utf-8")
                    model.save_weights(str(run_dir / "best_model.converted.weights.h5"))
                    raise RuntimeError("Spremljeni su zasebni arhitektura i težine; .keras nije nastao.") from second
            checked = keras.models.load_model(str(temp_path), compile=False,
                                              custom_objects=custom_objects)
            del checked
            temp_path.replace(out_path)
            print(f"Konvertirano: {weights_path} -> {out_path}")
            return "konvertirano"
    except (FileNotFoundError, ValueError, AttributeError, RuntimeError, OSError, TypeError) as exc:
        warnings.warn(f"Konverzija nije uspjela za {run_dir}: {type(exc).__name__}: {exc}")
        return "neuspješno"
    finally:
        if model is not None:
            del model
        keras.backend.clear_session()
        gc.collect()


def main():
    counts = {"konvertirano": 0, "preskočeno": 0, "neuspješno": 0}
    generated = []
    objects = custom_mtl_objects()

    for config in MANDATORY_CONFIGS:
        for seed in MANDATORY_SEEDS:
            run_dir = RUNS / "mtl" / config / f"seed_{seed}"
            status = convert_one(run_dir, lambda seed=seed: build_mtl(seed), objects)
            counts[status] += 1
            if status == "konvertirano":
                generated.append(run_dir / "best_model.keras")

    for seed in GAZE_SEEDS:
        run_dir = RUNS / "gaze_only" / f"seed_{seed}"
        status = convert_one(run_dir, build_gaze_model, objects)
        counts[status] += 1
        if status == "konvertirano":
            generated.append(run_dir / "best_model.keras")

    print("\n=== Sažetak konverzije ===")
    print(f"Konvertirano: {counts['konvertirano']}")
    print(f"Preskočeno: {counts['preskočeno']}")
    print(f"Neuspješno: {counts['neuspješno']}")
    print("Novo generirane .keras datoteke:")
    for path in generated:
        print(f"  {path}")
    print("Sada pokreni evaluate.ipynb ćeliju 6 za latenciju.")


main()

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3090, compute capability 8.6
TensorFlow 2.11.0; GPU: 1; policy: <Policy "mixed_float16">
decoder_features: resizing_2 (None, 128, 128, 12)
encoder_features: feature_extractor (None, 4, 4, 576)
  H5 dataseta: 228
  feature_extractor/Conv/kernel <- feature_extractor/Conv/kernel (3, 3, 3, 16)
  feature_extractor/Conv/BatchNorm/gamma <- feature_extractor/Conv/BatchNorm/gamma (16,)
  feature_extractor/Conv/BatchNorm/beta <- feature_extractor/Conv/BatchNorm/beta (16,)
  feature_extractor/Conv/BatchNorm/moving_mean <- feature_extractor/Conv/BatchNorm/moving_mean (16,)
  feature_extractor/Conv/BatchNorm/moving_variance <- feature_extractor/Conv/BatchNorm/moving_variance (16,)
  feature_extractor/expanded_conv/depthwise/depthwise_kernel <- feature_extractor/expanded_conv/depthwise/depth

  Test forward pass OK, izlazi: {'seg': (1, 128, 128, 3), 'gaze_output': (1, 3)}
Konvertirano: /home/fipulab/anastazija/runs/mtl/mtl_full/seed_42/best_model.weights.h5 -> /home/fipulab/anastazija/runs/mtl/mtl_full/seed_42/best_model.keras
decoder_features: resizing_2 (None, 128, 128, 12)
encoder_features: feature_extractor (None, 4, 4, 576)
  H5 dataseta: 228
  feature_extractor/Conv/kernel <- feature_extractor/Conv/kernel (3, 3, 3, 16)
  feature_extractor/Conv/BatchNorm/gamma <- feature_extractor/Conv/BatchNorm/gamma (16,)
  feature_extractor/Conv/BatchNorm/beta <- feature_extractor/Conv/BatchNorm/beta (16,)
  feature_extractor/Conv/BatchNorm/moving_mean <- feature_extractor/Conv/BatchNorm/moving_mean (16,)
  feature_extractor/Conv/BatchNorm/moving_variance <- feature_extractor/Conv/BatchNorm/moving_variance (16,)
  feature_extractor/expanded_conv/depthwise/depthwise_kernel <- feature_extractor/expanded_conv/depthwise/depthwise_kernel (3, 3, 16, 1)
  feature_extractor/expanded_conv/de

Konvertirano: /home/fipulab/anastazija/runs/mtl/mtl_full/seed_123/best_model.weights.h5 -> /home/fipulab/anastazija/runs/mtl/mtl_full/seed_123/best_model.keras
decoder_features: resizing_2 (None, 128, 128, 12)
encoder_features: feature_extractor (None, 4, 4, 576)
  H5 dataseta: 228
  feature_extractor/Conv/kernel <- feature_extractor/Conv/kernel (3, 3, 3, 16)
  feature_extractor/Conv/BatchNorm/gamma <- feature_extractor/Conv/BatchNorm/gamma (16,)
  feature_extractor/Conv/BatchNorm/beta <- feature_extractor/Conv/BatchNorm/beta (16,)
  feature_extractor/Conv/BatchNorm/moving_mean <- feature_extractor/Conv/BatchNorm/moving_mean (16,)
  feature_extractor/Conv/BatchNorm/moving_variance <- feature_extractor/Conv/BatchNorm/moving_variance (16,)
  feature_extractor/expanded_conv/depthwise/depthwise_kernel <- feature_extractor/expanded_conv/depthwise/depthwise_kernel (3, 3, 16, 1)
  feature_extractor/expanded_conv/depthwise/BatchNorm/gamma <- feature_extractor/expanded_conv/depthwise/BatchNorm/

Konvertirano: /home/fipulab/anastazija/runs/mtl/mtl_full/seed_2024/best_model.weights.h5 -> /home/fipulab/anastazija/runs/mtl/mtl_full/seed_2024/best_model.keras
decoder_features: resizing_2 (None, 128, 128, 12)
encoder_features: feature_extractor (None, 4, 4, 576)
  H5 dataseta: 228
  feature_extractor/Conv/kernel <- feature_extractor/Conv/kernel (3, 3, 3, 16)
  feature_extractor/Conv/BatchNorm/gamma <- feature_extractor/Conv/BatchNorm/gamma (16,)
  feature_extractor/Conv/BatchNorm/beta <- feature_extractor/Conv/BatchNorm/beta (16,)
  feature_extractor/Conv/BatchNorm/moving_mean <- feature_extractor/Conv/BatchNorm/moving_mean (16,)
  feature_extractor/Conv/BatchNorm/moving_variance <- feature_extractor/Conv/BatchNorm/moving_variance (16,)
  feature_extractor/expanded_conv/depthwise/depthwise_kernel <- feature_extractor/expanded_conv/depthwise/depthwise_kernel (3, 3, 16, 1)
  feature_extractor/expanded_conv/depthwise/BatchNorm/gamma <- feature_extractor/expanded_conv/depthwise/BatchNor

Konvertirano: /home/fipulab/anastazija/runs/mtl/mtl_wseg0/seed_42/best_model.weights.h5 -> /home/fipulab/anastazija/runs/mtl/mtl_wseg0/seed_42/best_model.keras
decoder_features: resizing_2 (None, 128, 128, 12)
encoder_features: feature_extractor (None, 4, 4, 576)
  H5 dataseta: 228
  feature_extractor/Conv/kernel <- feature_extractor/Conv/kernel (3, 3, 3, 16)
  feature_extractor/Conv/BatchNorm/gamma <- feature_extractor/Conv/BatchNorm/gamma (16,)
  feature_extractor/Conv/BatchNorm/beta <- feature_extractor/Conv/BatchNorm/beta (16,)
  feature_extractor/Conv/BatchNorm/moving_mean <- feature_extractor/Conv/BatchNorm/moving_mean (16,)
  feature_extractor/Conv/BatchNorm/moving_variance <- feature_extractor/Conv/BatchNorm/moving_variance (16,)
  feature_extractor/expanded_conv/depthwise/depthwise_kernel <- feature_extractor/expanded_conv/depthwise/depthwise_kernel (3, 3, 16, 1)
  feature_extractor/expanded_conv/depthwise/BatchNorm/gamma <- feature_extractor/expanded_conv/depthwise/BatchNorm/

Konvertirano: /home/fipulab/anastazija/runs/mtl/mtl_wseg0/seed_123/best_model.weights.h5 -> /home/fipulab/anastazija/runs/mtl/mtl_wseg0/seed_123/best_model.keras
decoder_features: resizing_2 (None, 128, 128, 12)
encoder_features: feature_extractor (None, 4, 4, 576)
  H5 dataseta: 228
  feature_extractor/Conv/kernel <- feature_extractor/Conv/kernel (3, 3, 3, 16)
  feature_extractor/Conv/BatchNorm/gamma <- feature_extractor/Conv/BatchNorm/gamma (16,)
  feature_extractor/Conv/BatchNorm/beta <- feature_extractor/Conv/BatchNorm/beta (16,)
  feature_extractor/Conv/BatchNorm/moving_mean <- feature_extractor/Conv/BatchNorm/moving_mean (16,)
  feature_extractor/Conv/BatchNorm/moving_variance <- feature_extractor/Conv/BatchNorm/moving_variance (16,)
  feature_extractor/expanded_conv/depthwise/depthwise_kernel <- feature_extractor/expanded_conv/depthwise/depthwise_kernel (3, 3, 16, 1)
  feature_extractor/expanded_conv/depthwise/BatchNorm/gamma <- feature_extractor/expanded_conv/depthwise/BatchNor

Konvertirano: /home/fipulab/anastazija/runs/mtl/mtl_wseg0/seed_2024/best_model.weights.h5 -> /home/fipulab/anastazija/runs/mtl/mtl_wseg0/seed_2024/best_model.keras
decoder_features: resizing_2 (None, 128, 128, 12)
encoder_features: feature_extractor (None, 4, 4, 576)
  H5 dataseta: 228
  feature_extractor/Conv/kernel <- feature_extractor/Conv/kernel (3, 3, 3, 16)
  feature_extractor/Conv/BatchNorm/gamma <- feature_extractor/Conv/BatchNorm/gamma (16,)
  feature_extractor/Conv/BatchNorm/beta <- feature_extractor/Conv/BatchNorm/beta (16,)
  feature_extractor/Conv/BatchNorm/moving_mean <- feature_extractor/Conv/BatchNorm/moving_mean (16,)
  feature_extractor/Conv/BatchNorm/moving_variance <- feature_extractor/Conv/BatchNorm/moving_variance (16,)
  feature_extractor/expanded_conv/depthwise/depthwise_kernel <- feature_extractor/expanded_conv/depthwise/depthwise_kernel (3, 3, 16, 1)
  feature_extractor/expanded_conv/depthwise/BatchNorm/gamma <- feature_extractor/expanded_conv/depthwise/BatchN

Konvertirano: /home/fipulab/anastazija/runs/mtl/mtl_pupil/seed_42/best_model.weights.h5 -> /home/fipulab/anastazija/runs/mtl/mtl_pupil/seed_42/best_model.keras
decoder_features: resizing_2 (None, 128, 128, 12)
encoder_features: feature_extractor (None, 4, 4, 576)
  H5 dataseta: 228
  feature_extractor/Conv/kernel <- feature_extractor/Conv/kernel (3, 3, 3, 16)
  feature_extractor/Conv/BatchNorm/gamma <- feature_extractor/Conv/BatchNorm/gamma (16,)
  feature_extractor/Conv/BatchNorm/beta <- feature_extractor/Conv/BatchNorm/beta (16,)
  feature_extractor/Conv/BatchNorm/moving_mean <- feature_extractor/Conv/BatchNorm/moving_mean (16,)
  feature_extractor/Conv/BatchNorm/moving_variance <- feature_extractor/Conv/BatchNorm/moving_variance (16,)
  feature_extractor/expanded_conv/depthwise/depthwise_kernel <- feature_extractor/expanded_conv/depthwise/depthwise_kernel (3, 3, 16, 1)
  feature_extractor/expanded_conv/depthwise/BatchNorm/gamma <- feature_extractor/expanded_conv/depthwise/BatchNorm/

Konvertirano: /home/fipulab/anastazija/runs/mtl/mtl_pupil/seed_123/best_model.weights.h5 -> /home/fipulab/anastazija/runs/mtl/mtl_pupil/seed_123/best_model.keras
decoder_features: resizing_2 (None, 128, 128, 12)
encoder_features: feature_extractor (None, 4, 4, 576)
  H5 dataseta: 228
  feature_extractor/Conv/kernel <- feature_extractor/Conv/kernel (3, 3, 3, 16)
  feature_extractor/Conv/BatchNorm/gamma <- feature_extractor/Conv/BatchNorm/gamma (16,)
  feature_extractor/Conv/BatchNorm/beta <- feature_extractor/Conv/BatchNorm/beta (16,)
  feature_extractor/Conv/BatchNorm/moving_mean <- feature_extractor/Conv/BatchNorm/moving_mean (16,)
  feature_extractor/Conv/BatchNorm/moving_variance <- feature_extractor/Conv/BatchNorm/moving_variance (16,)
  feature_extractor/expanded_conv/depthwise/depthwise_kernel <- feature_extractor/expanded_conv/depthwise/depthwise_kernel (3, 3, 16, 1)
  feature_extractor/expanded_conv/depthwise/BatchNorm/gamma <- feature_extractor/expanded_conv/depthwise/BatchNor

Konvertirano: /home/fipulab/anastazija/runs/mtl/mtl_pupil/seed_2024/best_model.weights.h5 -> /home/fipulab/anastazija/runs/mtl/mtl_pupil/seed_2024/best_model.keras
Već postoji, preskačem: /home/fipulab/anastazija/runs/gaze_only/seed_42/best_model.keras
Već postoji, preskačem: /home/fipulab/anastazija/runs/gaze_only/seed_123/best_model.keras
Već postoji, preskačem: /home/fipulab/anastazija/runs/gaze_only/seed_2024/best_model.keras
Već postoji, preskačem: /home/fipulab/anastazija/runs/gaze_only/seed_7/best_model.keras
Već postoji, preskačem: /home/fipulab/anastazija/runs/gaze_only/seed_13/best_model.keras

=== Sažetak konverzije ===
Konvertirano: 9
Preskočeno: 5
Neuspješno: 0
Novo generirane .keras datoteke:
  /home/fipulab/anastazija/runs/mtl/mtl_full/seed_42/best_model.keras
  /home/fipulab/anastazija/runs/mtl/mtl_full/seed_123/best_model.keras
  /home/fipulab/anastazija/runs/mtl/mtl_full/seed_2024/best_model.keras
  /home/fipulab/anastazija/runs/mtl/mtl_wseg0/seed_42/best_model.keras
